In [ ]:
# Importing needed code

from math import exp
from pathlib import Path
from typing import Callable, TypeVar, Any
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.signal import deconvolve, convolve

from data_processing.arc_paths import (
    get_parq_root, get_exp_root, INPUT_DATA_FOLDER, get_report_root
)
from data_processing.dataframe_validation import DetectorDataframeColumn
from data_processing.experiment_data_keys import ExperimentDataKey
from data_processing.loading.dataframe_loading import load_psd
from data_processing.loading.timetag_processing import calculate_timetag_hours
from data_processing.processing.bimodal_fitting import (
    get_psd_energy_histogram,
    scan_histogram_slices,
    find_failed_slices,
    BimodalBounds,
    BimodalParams
)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.figure_of_merit import gaussian
from data_processing.processing.neutron_classification import classify
from data_processing.processing.neutron_window_generation import (
    generate_nasa_neutron_window,
    generate_n_distro_neutron_window
)
from data_processing.reporting.plotting import plot_classification
from data_processing.helpers.stop_jupyter import stop

In [ ]:
exp_id = "ID-479"
file_number = "00001"
file_name = f"caen_psd_SData_{exp_id}_{file_number}.parquet"
file_path = get_parq_root(exp_id) / file_name

In [ ]:
df = pd.read_parquet(file_path)
df

In [ ]:
time_series = df["TIMETAG"].astype(np.int64)
time_max = time_series.max()
time_min = time_series.min()
duration = time_max - time_min
duration * 1e-12

In [ ]:
desired_time = 656 * 1e12  # picoseconds
desired_min = time_min
desired_max = desired_min + desired_time
within_desired_time = time_series[time_series.between(desired_min, desired_max)]
within_desired_time.shape

In [ ]:
within_desired_time

In [ ]:
(within_desired_time.max() - within_desired_time.min()) * 1e-12